# OpenBind-HIPPO

- **Target: A71EV2A**
- **Cycle: 01**

## Prep

- [x] Downloaded A71EV2A with fragalysis_download.ipynb
- [x] Bulkdock setup: `python -m bulkdock setup A71EV2A`
- [x] Copy aligned files to here: `cp -rv $BULK/TARGETS/XX01ZVNS2B/aligned_files .`

## Imports

In [1]:
%load_ext autoreload
%autoreload 2
import hippo
import mrich
from mrich import print
from pathlib import Path
from os import environ
import shutil

# Config

In [2]:
target_name = "A71EV2A"
target_dir = Path(environ["BULK"]) / "TARGETS" / target_name
cycle_name = "cycle_01"
cycle_dir = Path(cycle_name)
aligned_dir = target_dir / "aligned_files"

## Animal

In [3]:
animal = hippo.HIPPO(target_name, target_dir / f"{target_name}.sqlite")

 Creating HIPPO animal

name = A71EV2A

db_path = /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/A71EV2A/A71EV2A.sqlite

DEBUG: hippo.Database.__init__()

DEBUG: Database.path = /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/A71EV2A/A71EV2A.sqlite

DEBUG: hippo.Database.connect()

DEBUG: sqlite3.version='2.6.0'

 Success  Database connected @ /opt/xchem-fragalysis-2/maxwin/BulkDock/TARGETS/A71EV2A/A71EV2A.sqlite!

 Success  Initialised animal HIPPO("A71EV2A")!

## Merging

Run merging algorithms on all poses tagged 'hits'

In [4]:
merge_input_poses = animal.poses(tag="hits")
merge_input_poses

poses tagged "hits": {P × 645}

### Create inputs

- CSV input for Knitwork
- SDF of hits to merge for Knitwork and Fragmenstein
- Protonated PDBs for Knitwork
- Reference PDB for Fragmenstein

In [5]:
# output directories
knitwork_out_dir = cycle_dir / "knitwork"
knitwork_out_dir.mkdir(parents=True, exist_ok=True)
fragmenstein_out_dir = cycle_dir / "fragmenstein"
fragmenstein_out_dir.mkdir(parents=True, exist_ok=True)

In [6]:
# knitwork CSV
knitwork_out_dir = cycle_dir / "knitwork"
knitwork_out_dir.mkdir(parents=True, exist_ok=True)
merge_input_poses.to_knitwork(knitwork_out_dir / f"{cycle_name}_input.csv", path_root=knitwork_out_dir, aligned_files_dir="aligned_files")

out_path = /opt/xchem-fragalysis-2/maxwin/openbind-hippo/a71ev2a/cycle_01/knitwork/cycle_01_input.csv

path_root = /opt/xchem-fragalysis-2/maxwin/openbind-hippo/a71ev2a/cycle_01/knitwork

aligned_files_dir = aligned_files

 DISK  Writing /opt/xchem-fragalysis-2/maxwin/openbind-hippo/a71ev2a/cycle_01/knitwork/cycle_01_input.csv...

In [7]:
# reference apo PDB for Fragmenstein
ref_pose = merge_input_poses[0]
mrich.var("ref_pose", ref_pose)
shutil.copy(ref_pose.apo_path, fragmenstein_out_dir)

ref_pose = C1->P1: "A5666a"

'cycle_01/fragmenstein/A5666a_apo-desolv.pdb'

In [8]:
# SDF of hits
out_dir = cycle_dir
out_dir.mkdir(parents=True, exist_ok=True)
merge_input_poses.write_sdf(cycle_dir / f"{cycle_name}_hits.sdf")

Output()

 DISK  Writing cycle_01/cycle_01_hits.sdf...

### Run Fragmenstein

```
cd cycle_01/fragmenstein
sbatch --job-name "a71ev2a_fragmenstein" --mem 16000 $HOME2/slurm/run_bash_with_conda.sh run_fragmenstein.sh
```